<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_09_post_tuning/stage_09_post_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_09 - T2 SEQ2ONE - POST TUNING**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [37]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-23 19:21:10,278 | INFO | Environment initialized


## **2. Acceso a drive**

In [38]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

2026-04-23 19:21:12,438 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [39]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
  "t2_p40_h30",
  "t2_p40_h60",
  "t2_p50_h30",
]

# Tamaños de ventana
WINDOW_SIZES = [30]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
        "ema_60",
        "roc_60",
        "roc_30",
        "stoch_k_30",
        "mom_5",
        "atr_norm_10",
        "macd"
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-23 19:21:12,446 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-23 19:21:12,447 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-23 19:21:12,448 | INFO | Configuración de experimento cargada
2026-04-23 19:21:12,449 | INFO | Targets: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
2026-04-23 19:21:12,450 | INFO | Window sizes: [30]


In [40]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_mnq_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:10]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[30]["t2_p40_h30"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-23 19:21:12,462 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-23 19:21:12,466 | INFO | Windows OK      : 9
2026-04-23 19:21:12,467 | INFO | Windows missing : 0
2026-04-23 19:21:12,468 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl
2026-04-23 19:21:12,469 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [41]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [42]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [43]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_p40_h30'
        - 't2_p40_h60'
        - 't2_p50_h30'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon_str = target.split("_")[-1]   # ej: "h30"
        horizon = int(horizon_str.replace("h", ""))
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }

### **4.4. Creación de bundles T2**

In [44]:
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundles : dict
        bundles[target] -> bundle dict
    """

    bundles = {}

    # --------------------------
    # Construcción
    # --------------------------
    for target in targets:
        bundles[target] = load_windows_and_scaler(
            window_size=window_size,
            target=target,
            windows_paths=windows_paths,
            scaler_path=scaler_path,
        )

    # --------------------------
    # Verificación rápida
    # --------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    for target in targets:
        b = bundles[target]

        print(f"\nTARGET: {target}")
        print("Train :", b["train"]["X"].shape, b["train"]["y"].shape)
        print("Valid :", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print("Test  :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print("Scaler:", type(b["scaler"]).__name__)

    return bundles

In [45]:
bundles_L30 = create_bundles(window_size=30)

2026-04-23 19:21:12,607 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 19:21:12,608 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 19:21:12,629 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 19:21:12,629 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 19:21:12,652 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 19:21:12,652 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 19:21:12,658 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 19:21:12,658 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-23 19:21:12,751 | INFO | Loaded: windows_t2_p40_h60_train.npz
2026-04-23 19:21:12,752 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 19:21:12,774 | INFO | Loaded: windows_t2_p40_h60_valid.npz
2026-04-23 19:21:12,775 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 19:21:12,795 | INFO | Loaded: windows_t2_p4


WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p40_h60
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler


Como acceder a las ventanas X e y:

```python
bundle_p40_h30 = bundles_L30["t2_p40_h30"]

X_train = bundle_p40_h30["train"]["X"]
y_train = bundle_p40_h30["train"]["y"]

X_valid = bundle_p40_h30["valid"]["X"]
y_valid = bundle_p40_h30["valid"]["y"]

X_test = bundle_p40_h30["test"]["X"]
y_test = bundle_p40_h30["test"]["y"]

scaler = bundle_p40_h30["scaler"]

print("Target :", bundle_p40_h30["target"])
print("Horizon:", bundle_p40_h30["horizon"])
print("Train  :", X_train.shape, y_train.shape)
print("Valid  :", X_valid.shape, y_valid.shape)
print("Test   :", X_test.shape, y_test.shape)
print("Scaler :", type(scaler).__name__)
```




In [46]:
bundles_L30

{'t2_p40_h30': {'window_size': 30,
  'target': 't2_p40_h30',
  'horizon': 30,
  'paths': {'train': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_train.npz',
   'valid': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_valid.npz',
   'test': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_test.npz',
   'scaler': '/content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl'},
  'scaler': StandardScaler(),
  'train': {'X': array([[[-0.11680385, -0.06583842, -0.2238865 , ..., -0.04129868,
            -1.507039  , -0.02778307],
           [ 0.0909589 ,  0.05818945, -0.07248875, ...,  0.3872164 ,
            -1.3723946 ,  0.08631181],
           [-0.18229802, -0.13053213, -0.21077001, ..., -0.1151728 ,
            -1.2194692 ,  0.02307655],
           ...,
           [ 0.40151003,  0.28442496,  0.59289074, ...,  0.47497907,
            -1.0690751 , -0.19294474],
         

### **4.5. Preparación de inputs según el tipo de modelo**

In [47]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

Estos sanity checks sirven para verificar, antes de entrenar, que los datos cargados tengan la estructura correcta y no vengan con errores silenciosos.

En concreto, comprueban que:

- X tenga el formato esperado: 3D (n, seq_len, n_features) o 2D (n, d_flat)
- y tenga forma válida para clasificación seq2one
- X e y tengan la misma cantidad de muestras
- no haya NaN ni inf
- las dimensiones sean consistentes entre train, valid y test
- exista más de una clase en y

Nos conviene tenerlos, porque ayudan a detectar errores de shape o de datos antes de llegar al entrenamiento.

In [48]:
from __future__ import annotations

from typing import Any, Optional, Tuple, Dict, Mapping
import numpy as np


# ============================================================
# SANITY CHECKS PARA DATASETS SEQ2ONE (T2)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía
    ni contenga NaN/inf.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(
    y: np.ndarray,
    *,
    name: str = "y",
    allow_seq_inputs_take_last: bool = False,
) -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)
    - (n, seq_len)       si allow_seq_inputs_take_last=True
    - (n, seq_len, 1)    si allow_seq_inputs_take_last=True
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    if allow_seq_inputs_take_last:
        if y.ndim == 2 and y.shape[1] > 1:
            return y[:, -1]

        if y.ndim == 3 and y.shape[2] == 1:
            return y[:, -1, 0]

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1)"
        f"{' o secuencial si allow_seq_inputs_take_last=True' if allow_seq_inputs_take_last else ''}. "
        f"Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )


def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)
      - y secuencial, opcionalmente, tomando el último valor

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: no aplica directamente.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D.
    allow_seq_inputs_take_last:
        Si y viene como secuencia, toma el último valor.
    """
    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    y = _normalize_y_seq2one(
        y,
        name=f"y[{split_name}]",
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
    )

    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info


def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_p40_h30",
      "horizon": 30,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Si no se pasan expected_*, usa TRAIN como referencia.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_bundles_seq2one(
    bundles: Mapping[str, Dict[str, Any]],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para todos los bundles de un diccionario:

    bundles[target] -> bundle
    """
    results = {}

    for target, bundle in bundles.items():
        results[target] = run_sanity_checks_for_bundle_seq2one(
            bundle,
            tag=target,
            verbose=verbose,
        )

    return results

In [49]:
sanity_results = run_sanity_checks_all_bundles_seq2one(
    bundles_L30,
    verbose=True,
)

[sanity_check_seq2one] train_t2_p40_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h30 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h30 | target=t2_p40_h30 | horizon=30
[sanity_check_seq2one] train_t2_p40_h60 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h60 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h60 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h60 | target=t2_p40_h60 | horizon=60
[sanity_check_seq2one] train_t2_p50_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p50_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | c

## **6. Módulo de métricas T2**

In [50]:
# ================================
# Setup para importar módulos del proyecto
# ================================

import sys
import importlib

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

# asegurar que metrics es paquete
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

from metrics.classification_probabilities import (
    compute_probabilistic_outputs,
    apply_decision_rule,
)

print("Módulos importados correctamente")

Módulos importados correctamente


In [51]:
# ================================
# Utilidades: outputs -> DataFrame
# ================================

from __future__ import annotations

import pandas as pd
from typing import Any


def classification_metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output de compute_classification_metrics(...)
    en una fila de DataFrame.

    Usa metrics_to_flat_dict(...) para aplanar la salida
    del módulo classification_metrics y luego agrega metadata
    del experimento.
    """
    flat_metrics = metrics_to_flat_dict(metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_metrics,
    }

    return pd.DataFrame([row])


def _flatten_dict(
    d: dict[str, Any],
    *,
    parent_key: str = "",
    sep: str = "_",
) -> dict[str, Any]:
    """
    Aplana un diccionario arbitrario de forma recursiva.

    Ejemplo:
    {"a": {"b": 1}} -> {"a_b": 1}
    """
    items: dict[str, Any] = {}

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)

        if isinstance(v, dict):
            items.update(_flatten_dict(v, parent_key=new_key, sep=sep))
        else:
            items[new_key] = v

    return items


def probabilities_metrics_to_df(
    prob_metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output del módulo classification_probabilities
    en una fila de DataFrame.

    Como la estructura puede variar según la implementación,
    se aplana recursivamente el diccionario y luego se agrega
    metadata del experimento.
    """
    flat_prob_metrics = _flatten_dict(prob_metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_prob_metrics,
    }

    return pd.DataFrame([row])


logger.info("Utilidades de exportación a DataFrame cargadas")

2026-04-23 19:21:13,055 | INFO | Utilidades de exportación a DataFrame cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

### **7.1. Persistencia de métricas y probabilidades**

In [52]:
# ================================
# Persistencia de métricas de clasificación
# ================================

from pathlib import Path
import pandas as pd


def load_classification_metrics_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_best_model" / "classification_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen métricas previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_best_model" / "classification_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path


# ================================
# Persistencia de probabilidades / decisión
# ================================

def load_classification_probabilities_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_best_model" / "classification_probabilities",
) -> pd.DataFrame:
    """
    Carga resultados probabilísticos si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando probabilidades desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen probabilidades previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_probabilities(
    df_probabilities: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_best_model" / "classification_probabilities",
) -> Path:
    """
    Guarda un DataFrame de probabilidades / decisión en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"
    df_probabilities.to_parquet(out_path, index=False)

    logger.info(f"Probabilidades guardadas en: {out_path}")

    return out_path

### **7.2. Persistencia de modelos**

In [53]:
OUTPUT_DIR_BEST_MODEL = Path("/content/drive/MyDrive/neural_profit/best_models")

In [54]:
from pathlib import Path
import json
import joblib
import pandas as pd


def save_model_artifacts(
    *,
    models: dict,
    output_dir,
    experiment_name: str,
    metrics: pd.DataFrame | None = None,
    probabilities: pd.DataFrame | None = None,
    metadata: dict | None = None,
    overwrite: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Guarda artefactos de entrenamiento de forma genérica para cualquier modelo.

    Estructura:
    output_dir/
        experiment_name/
            models/
                *.joblib
            metrics.parquet
            probabilities.parquet
            metadata.json
    """

    if not isinstance(models, dict) or len(models) == 0:
        raise ValueError("`models` debe ser un diccionario no vacío.")

    output_dir = Path(output_dir)
    artifact_dir = output_dir / experiment_name
    models_dir = artifact_dir / "models"

    artifact_dir.mkdir(parents=True, exist_ok=True)
    models_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = {
        "artifact_dir": str(artifact_dir),
        "models": {},
        "metrics": None,
        "probabilities": None,
        "metadata": None,
    }

    # ==================================================
    # 1) Guardar modelos
    # ==================================================
    for model_key, model_obj in models.items():
        safe_name = str(model_key).replace("/", "__").replace("\\", "__").replace(" ", "_")
        model_path = models_dir / f"{safe_name}.joblib"

        if model_path.exists() and not overwrite:
            raise FileExistsError(f"El modelo ya existe y overwrite=False: {model_path}")

        joblib.dump(model_obj, model_path)
        saved_paths["models"][model_key] = str(model_path)

    # ==================================================
    # 2) Guardar métricas
    # ==================================================
    if metrics is not None:
        metrics_path = artifact_dir / "metrics.parquet"
        if metrics_path.exists() and not overwrite:
            raise FileExistsError(f"El archivo ya existe y overwrite=False: {metrics_path}")
        metrics.to_parquet(metrics_path, index=False)
        saved_paths["metrics"] = str(metrics_path)

    # ==================================================
    # 3) Guardar probabilidades
    # ==================================================
    if probabilities is not None:
        probabilities_path = artifact_dir / "probabilities.parquet"
        if probabilities_path.exists() and not overwrite:
            raise FileExistsError(f"El archivo ya existe y overwrite=False: {probabilities_path}")
        probabilities.to_parquet(probabilities_path, index=False)
        saved_paths["probabilities"] = str(probabilities_path)

    # ==================================================
    # 4) Guardar metadata
    # ==================================================
    metadata_to_save = metadata.copy() if metadata is not None else {}
    metadata_to_save["experiment_name"] = experiment_name
    metadata_to_save["n_models"] = len(models)
    metadata_to_save["model_keys"] = list(models.keys())

    metadata_path = artifact_dir / "metadata.json"
    if metadata_path.exists() and not overwrite:
        raise FileExistsError(f"El archivo ya existe y overwrite=False: {metadata_path}")

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata_to_save, f, ensure_ascii=False, indent=2)

    saved_paths["metadata"] = str(metadata_path)

    if verbose:
        print("\n" + "=" * 80)
        print("ARTEFACTOS GUARDADOS")
        print("=" * 80)
        print(f"artifact_dir   = {artifact_dir}")
        print(f"n_models       = {len(models)}")
        print(f"metrics_saved  = {metrics is not None}")
        print(f"prob_saved     = {probabilities is not None}")
        print(f"metadata_saved = True")

    return saved_paths

### **7.3. Verificación de existencia de experimento**

In [55]:
from pathlib import Path


def make_model_key(model_name: str, target: str, window_size: int) -> str:
    """
    Construye la clave estándar de un modelo entrenado.
    """
    return f"{model_name}__{target}__L{int(window_size)}"


def make_expected_model_keys(
    *,
    model_name: str,
    targets: list[str],
    window_size: int,
) -> list[str]:
    """
    Genera las claves esperadas de modelos para un experimento.
    """
    return [
        make_model_key(model_name=model_name, target=target, window_size=window_size)
        for target in targets
    ]


def check_experiment_artifacts_exist(
    *,
    output_dir,
    experiment_name: str,
    expected_model_keys: list[str] | None = None,
    require_metrics: bool = True,
    require_probabilities: bool = True,
) -> dict:
    """
    Verifica si ya existen los artefactos de un experimento.

    Estructura esperada:
    output_dir/
        experiment_name/
            models/
                *.joblib
            metrics.parquet
            probabilities.parquet
            metadata.json
    """

    output_dir = Path(output_dir)
    artifact_dir = output_dir / experiment_name
    models_dir = artifact_dir / "models"

    metrics_path = artifact_dir / "metrics.parquet"
    probabilities_path = artifact_dir / "probabilities.parquet"
    metadata_path = artifact_dir / "metadata.json"

    metrics_exists = metrics_path.exists()
    probabilities_exists = probabilities_path.exists()
    metadata_exists = metadata_path.exists()
    models_dir_exists = models_dir.exists()

    existing_model_files = []
    if models_dir_exists:
        existing_model_files = sorted([p.name for p in models_dir.glob("*.joblib")])

    missing_model_keys = []
    if expected_model_keys is not None:
        for model_key in expected_model_keys:
            safe_name = (
                str(model_key)
                .replace("/", "__")
                .replace("\\", "__")
                .replace(" ", "_")
            )
            model_path = models_dir / f"{safe_name}.joblib"
            if not model_path.exists():
                missing_model_keys.append(model_key)

    models_ok = models_dir_exists
    if expected_model_keys is not None:
        models_ok = models_ok and (len(missing_model_keys) == 0)

    metrics_ok = metrics_exists if require_metrics else True
    probabilities_ok = probabilities_exists if require_probabilities else True

    exists_all = artifact_dir.exists() and models_ok and metrics_ok and probabilities_ok

    return {
        "exists_all": exists_all,
        "artifact_dir": str(artifact_dir),
        "models_dir": str(models_dir),
        "metrics_path": str(metrics_path),
        "probabilities_path": str(probabilities_path),
        "metadata_path": str(metadata_path),
        "models_dir_exists": models_dir_exists,
        "metrics_exists": metrics_exists,
        "probabilities_exists": probabilities_exists,
        "metadata_exists": metadata_exists,
        "existing_model_files": existing_model_files,
        "missing_model_keys": missing_model_keys,
        "expected_model_keys": expected_model_keys if expected_model_keys is not None else [],
    }


def resolve_experiment_status(
    *,
    output_dir,
    experiment_name: str,
    model_name: str,
    targets: list[str],
    window_size: int,
    require_metrics: bool = True,
    require_probabilities: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Función genérica para cualquier modelo:
    - genera expected_model_keys
    - verifica si existen artefactos
    - define si hay que entrenar o se puede omitir

    Retorna un dict con:
    - should_skip
    - expected_model_keys
    - artifact_check
    - experiment_name
    """

    expected_model_keys = make_expected_model_keys(
        model_name=model_name,
        targets=targets,
        window_size=window_size,
    )

    artifact_check = check_experiment_artifacts_exist(
        output_dir=output_dir,
        experiment_name=experiment_name,
        expected_model_keys=expected_model_keys,
        require_metrics=require_metrics,
        require_probabilities=require_probabilities,
    )

    should_skip = artifact_check["exists_all"]

    if verbose:
        print("\n" + "=" * 80)
        print("EXPERIMENT CHECK")
        print("=" * 80)
        print(f"experiment_name       = {experiment_name}")
        print(f"model_name            = {model_name}")
        print(f"window_size           = L{int(window_size)}")
        print(f"targets               = {targets}")
        print(f"artifact_dir          = {artifact_check['artifact_dir']}")
        print(f"models_dir_exists     = {artifact_check['models_dir_exists']}")
        print(f"metrics_exists        = {artifact_check['metrics_exists']}")
        print(f"probabilities_exists  = {artifact_check['probabilities_exists']}")
        print(f"metadata_exists       = {artifact_check['metadata_exists']}")
        print(f"n_expected_models     = {len(expected_model_keys)}")
        print(f"n_missing_models      = {len(artifact_check['missing_model_keys'])}")
        print(f"should_skip           = {should_skip}")

        if artifact_check["missing_model_keys"]:
            print("\n[MISSING MODEL KEYS]")
            for key in artifact_check["missing_model_keys"]:
                print(f" - {key}")

    return {
        "should_skip": should_skip,
        "expected_model_keys": expected_model_keys,
        "artifact_check": artifact_check,
        "experiment_name": experiment_name,
    }

## **7.4. Carga de artefactos**

In [56]:
from pathlib import Path
import json
import joblib
import pandas as pd


def load_model_artifacts(
    *,
    output_dir,
    experiment_name: str,
    load_metrics: bool = True,
    load_probabilities: bool = True,
    load_metadata: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Carga artefactos previamente guardados por save_model_artifacts().

    Retorna
    -------
    dict con:
    - "models"
    - "metrics"
    - "probabilities"
    - "metadata"
    - "artifact_dir"
    """

    output_dir = Path(output_dir)
    artifact_dir = output_dir / experiment_name
    models_dir = artifact_dir / "models"

    if not artifact_dir.exists():
        raise FileNotFoundError(f"No existe el directorio del experimento: {artifact_dir}")

    if not models_dir.exists():
        raise FileNotFoundError(f"No existe el directorio de modelos: {models_dir}")

    # ==================================================
    # 1) Cargar modelos
    # ==================================================
    models = {}
    model_files = sorted(models_dir.glob("*.joblib"))

    if not model_files:
        raise FileNotFoundError(f"No se encontraron modelos .joblib en: {models_dir}")

    for model_path in model_files:
        model_key = model_path.stem
        models[model_key] = joblib.load(model_path)

    # ==================================================
    # 2) Cargar métricas
    # ==================================================
    metrics = pd.DataFrame()
    metrics_path = artifact_dir / "metrics.parquet"
    if load_metrics:
        if not metrics_path.exists():
            raise FileNotFoundError(f"No existe metrics.parquet: {metrics_path}")
        metrics = pd.read_parquet(metrics_path)

    # ==================================================
    # 3) Cargar probabilidades
    # ==================================================
    probabilities = pd.DataFrame()
    probabilities_path = artifact_dir / "probabilities.parquet"
    if load_probabilities:
        if not probabilities_path.exists():
            raise FileNotFoundError(f"No existe probabilities.parquet: {probabilities_path}")
        probabilities = pd.read_parquet(probabilities_path)

    # ==================================================
    # 4) Cargar metadata
    # ==================================================
    metadata = {}
    metadata_path = artifact_dir / "metadata.json"
    if load_metadata:
        if not metadata_path.exists():
            raise FileNotFoundError(f"No existe metadata.json: {metadata_path}")
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

    if verbose:
        print("\n" + "=" * 80)
        print("ARTEFACTOS CARGADOS")
        print("=" * 80)
        print(f"artifact_dir = {artifact_dir}")
        print(f"n_models     = {len(models)}")
        print(f"metrics_rows = {len(metrics)}")
        print(f"prob_rows    = {len(probabilities)}")

    return {
        "artifact_dir": str(artifact_dir),
        "models": models,
        "metrics": metrics,
        "probabilities": probabilities,
        "metadata": metadata,
    }

## **8. Gestión de dispositivo y memoria**

In [57]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [58]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-23 19:21:13,142 | INFO | Seeds fijadas en 42


# **10. Modelo LR (Logistic Regression)**

## **Hiperparámetros seleccionados**

In [65]:
# =========================================================
# Logistic Regression | Configuración final de hiperparámetros
# =========================================================

lr_config = {
    "model_name": "logistic_regression",

    "model_params": {
        "C": 0.01,
        "max_iter": 1000,
        "solver": "lbfgs",
        "multi_class": "multinomial",
        "class_weight": "balanced",
        "random_state": SEED,
        "input_mode": "2d_flat",
    },

    "decision_rules": {
        "balanced": {
            "threshold_long": 0.40,
            "threshold_short": 0.45,
        },
        "conservative": {
            "threshold_long": 0.45,
            "threshold_short": 0.45,
        },
        "baseline": {
            "threshold_long": 0.40,
            "threshold_short": 0.40,
        },
    },
}

## **10.1. Función unitaria por bundle**

In [60]:
from sklearn.linear_model import LogisticRegression

def run_logistic_for_bundle_seq2one(
    bundle,
    *,
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    C=1.0,
    random_state=42,
    input_mode="2d_flat",
    class_weight=None,
    n_jobs=None,
    verbose=False,
):
    """
    Entrena Logistic Regression para un bundle seq2one
    usando TRAIN y genera predicciones sobre VALID.

    Esta función:
    - prepara X_train / X_valid
    - entrena el modelo
    - devuelve el modelo entrenado
    - devuelve predicciones de clase y probabilidades en VALID

    No aplica thresholds de decisión; eso se resuelve después
    en la etapa de evaluación operativa.
    """

    # =========================
    # 1. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    target = bundle.get("target")
    horizon = bundle.get("horizon")
    window_size = bundle.get("window_size")

    # =========================
    # 2. PREPARAR INPUT
    # =========================
    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)

    # =========================
    # 3. MODELO
    # =========================
    model = LogisticRegression(
        multi_class=multi_class,
        solver=solver,
        max_iter=max_iter,
        C=C,
        random_state=random_state,
        class_weight=class_weight,
        n_jobs=n_jobs,
    )

    # =========================
    # 4. TRAIN
    # =========================
    model.fit(X_train_model, y_train)

    # =========================
    # 5. PREDICT (VALID)
    # =========================
    y_pred_valid = model.predict(X_valid_model)
    y_proba_valid = model.predict_proba(X_valid_model)

    if verbose:
        print(
            f"[LOGISTIC] target={target} | horizon={horizon} | "
            f"window_size={window_size} | "
            f"X_train={X_train_model.shape} | X_valid={X_valid_model.shape} | "
            f"solver={solver} | C={C} | class_weight={class_weight}"
        )

    return {
        "model_name": "logistic_regression",
        "target": target,
        "horizon": horizon,
        "window_size": window_size,
        "input_mode": input_mode,
        "multi_class": multi_class,
        "solver": solver,
        "max_iter": max_iter,
        "C": C,
        "random_state": random_state,
        "class_weight": class_weight,
        "n_jobs": n_jobs,
        "X_train_shape": X_train_model.shape,
        "X_valid_shape": X_valid_model.shape,
        "classes_": model.classes_.tolist(),
        "model": model,
        "y_valid": y_valid,
        "y_pred_valid": y_pred_valid,
        "y_proba_valid": y_proba_valid,
    }

## **10.2. Función de evaluación sobre uno o más bundles**

In [61]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd


def eval_logistic_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    model_name: str = "logistic_regression",
    max_iter: int = 1000,
    random_state: int = 42,
    C: float = 1.0,
    multi_class: str = "multinomial",
    solver: str = "lbfgs",
    input_mode: str = "2d_flat",
    class_weight=None,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
    verbose: bool = False,
):
    """
    Evalúa Logistic Regression para uno o varios bundles seq2one
    usando VALID.

    Retorna
    -------
    dict con:
    - "models": dict de modelos entrenados
    - "metrics": DataFrame de métricas
    - "probabilities": DataFrame de outputs probabilísticos / decisión
    """

    # ==================================================
    # 1) Normalizar entrada
    # ==================================================
    if isinstance(bundles, dict):
        if "train" in bundles and "valid" in bundles:
            bundles_list: List[Dict[str, Any]] = [bundles]
        else:
            bundles_list = list(bundles.values())
    else:
        bundles_list = list(bundles)

    metrics_rows = []
    probabilities_rows = []
    models_dict = {}

    # ==================================================
    # 2) Iterar por bundles
    # ==================================================
    for bundle in bundles_list:
        target = bundle.get("target")
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"model={model_name} | "
                f"class_weight={class_weight}"
            )

        # ----------------------------------------------
        # 3) Entrenar + predecir
        # ----------------------------------------------
        preds = run_logistic_for_bundle_seq2one(
            bundle,
            multi_class=multi_class,
            solver=solver,
            max_iter=max_iter,
            C=C,
            random_state=random_state,
            input_mode=input_mode,
            class_weight=class_weight,
            verbose=False,
        )

        model_key = f"{model_name}__{target}__L{window_size}"
        models_dict[model_key] = preds["model"]

        y_true = preds["y_valid"]
        y_pred = preds["y_pred_valid"]
        y_proba = preds["y_proba_valid"]
        model_classes = preds["classes_"]

        expected_classes = [-1, 0, 1]
        if list(model_classes) != expected_classes:
            raise ValueError(
                f"Orden de clases inesperado para predict_proba. "
                f"Esperado={expected_classes}, obtenido={model_classes}"
            )

        experiment_meta = {
            "model": model_name,
            "split": "valid",
            "window_size": window_size,
            "target": target,
            "horizon": horizon,
            "class_weight_mode": "balanced" if class_weight == "balanced" else "none",
            "input_mode": input_mode,
            "C": C,
            "max_iter": max_iter,
            "multi_class": multi_class,
            "solver": solver,
            "random_state": random_state,
            "threshold_long": prob_threshold_long,
            "threshold_short": prob_threshold_short,
        }

        # ----------------------------------------------
        # 4) Métricas
        # ----------------------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split="valid",
            target=target,
            labels=expected_classes,
        )

        df_metrics_row = classification_metrics_to_df(
            metrics,
            model=model_name,
            split="valid",
            window_size=window_size,
            target=target,
            horizon=horizon,
        )

        for k, v in experiment_meta.items():
            df_metrics_row[k] = v

        metrics_rows.append(df_metrics_row)

        # ----------------------------------------------
        # 5) Probabilidades + decisión
        # ----------------------------------------------
        proba_df = compute_probabilistic_outputs(
            y_proba=y_proba,
            class_labels=expected_classes,
            y_true=y_true,
        )

        decision_df = apply_decision_rule(
            proba_df,
            long_class=1,
            short_class=-1,
            long_threshold=prob_threshold_long,
            short_threshold=prob_threshold_short,
        )

        overlap_cols = [c for c in decision_df.columns if c in proba_df.columns]
        if overlap_cols:
            decision_df = decision_df.drop(columns=overlap_cols)

        df_prob = pd.concat(
            [proba_df.reset_index(drop=True), decision_df.reset_index(drop=True)],
            axis=1,
        )

        for k, v in experiment_meta.items():
            df_prob[k] = v

        df_prob["class_labels"] = str(expected_classes)

        probabilities_rows.append(df_prob)

    # ==================================================
    # 6) Consolidar salida
    # ==================================================
    df_metrics_all = (
        pd.concat(metrics_rows, ignore_index=True)
        if metrics_rows else pd.DataFrame()
    )

    df_probabilities_all = (
        pd.concat(probabilities_rows, ignore_index=True)
        if probabilities_rows else pd.DataFrame()
    )

    return {
        "models": models_dict,
        "metrics": df_metrics_all,
        "probabilities": df_probabilities_all,
    }

## **10.3. Función orquestadora por `window_size`**

In [62]:
import gc
import pandas as pd


def run_logistic(
    window_size: int,
    *,
    targets: list[str] | None = None,
    experiment_name: str,
    output_dir,
    save_artifacts: bool = True,
    force_retrain: bool = False,
    verbose: bool = True,
    model_name: str = "logistic_regression",
    max_iter: int = 1000,
    random_state: int = 42,
    C: float = 1.0,
    multi_class: str = "multinomial",
    solver: str = "lbfgs",
    input_mode: str = "2d_flat",
    class_weight=None,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
):
    """
    Ejecuta Logistic Regression para una sola window_size
    sobre los targets T2 indicados.

    Flujo:
    - si existen artefactos y force_retrain=False -> carga y devuelve
    - si no existen -> entrena, evalúa, guarda y devuelve

    Retorna
    -------
    dict con:
    - "models"
    - "metrics"
    - "probabilities"
    - "metadata"
    - "artifact_dir" (si carga o guarda)
    """

    if targets is None:
        targets = TARGETS

    if not targets:
        raise ValueError("La lista de targets no puede estar vacía.")

    size = int(window_size)

    expected_model_keys = [
        f"{model_name}__{target}__L{size}"
        for target in targets
    ]

    # ==================================================
    # 0) Check artefactos existentes
    # ==================================================
    if not force_retrain:
        check = check_experiment_artifacts_exist(
            output_dir=output_dir,
            experiment_name=experiment_name,
            expected_model_keys=expected_model_keys,
            require_metrics=True,
            require_probabilities=True,
        )

        if check["exists_all"]:
            if verbose:
                print("\n" + "=" * 80)
                print("EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS")
                print("=" * 80)
                print(f"experiment_name = {experiment_name}")
                print(f"artifact_dir    = {check['artifact_dir']}")

            loaded = load_model_artifacts(
                output_dir=output_dir,
                experiment_name=experiment_name,
                load_metrics=True,
                load_probabilities=True,
                load_metadata=True,
                verbose=verbose,
            )

            return loaded

    bundles = None
    results = None

    try:
        # ==================================================
        # 1) Encabezado
        # ==================================================
        if verbose:
            print("\n" + "=" * 80)
            print(f"LOGISTIC REGRESSION | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"experiment_name = {experiment_name}")
            print(f"targets         = {targets}")
            print(f"class_weight    = {class_weight}")
            print(f"input_mode      = {input_mode}")
            print(f"C               = {C}")
            print(f"max_iter        = {max_iter}")
            print(f"solver          = {solver}")
            print(f"multi_class     = {multi_class}")
            print(f"random_state    = {random_state}")
            print(f"thr_long        = {prob_threshold_long}")
            print(f"thr_short       = {prob_threshold_short}")

        # ==================================================
        # 2) Construcción de bundles
        # ==================================================
        if verbose:
            print(f"\n[BUILD] L{size} | n_targets={len(targets)}")

        bundles = create_bundles(
            window_size=size,
            targets=targets,
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        # ==================================================
        # 3) Entrenamiento + evaluación VALID
        # ==================================================
        if verbose:
            print(
                f"\n[EVAL] L{size} | split=valid | "
                f"model={model_name} | class_weight={class_weight}"
            )

        results = eval_logistic_bundles(
            bundles=bundles,
            model_name=model_name,
            max_iter=max_iter,
            random_state=random_state,
            C=C,
            multi_class=multi_class,
            solver=solver,
            input_mode=input_mode,
            class_weight=class_weight,
            prob_threshold_long=prob_threshold_long,
            prob_threshold_short=prob_threshold_short,
            verbose=verbose,
        )

        models_dict = results["models"]

        df_metrics = results["metrics"]
        if not df_metrics.empty:
            sort_cols_metrics = [
                c for c in ["window_size", "target", "split", "horizon", "model"]
                if c in df_metrics.columns
            ]
            if sort_cols_metrics:
                df_metrics = (
                    df_metrics
                    .sort_values(sort_cols_metrics)
                    .reset_index(drop=True)
                )

        df_probabilities = results["probabilities"]
        if not df_probabilities.empty:
            sort_cols_prob = [
                c for c in ["window_size", "target", "split", "horizon", "model"]
                if c in df_probabilities.columns
            ]
            if sort_cols_prob:
                df_probabilities = (
                    df_probabilities
                    .sort_values(sort_cols_prob)
                    .reset_index(drop=True)
                )

        metadata = {
            "model_family": model_name,
            "targets": targets,
            "window_size": size,
            "C": C,
            "max_iter": max_iter,
            "solver": solver,
            "multi_class": multi_class,
            "input_mode": input_mode,
            "class_weight": class_weight,
            "random_state": random_state,
            "threshold_long": prob_threshold_long,
            "threshold_short": prob_threshold_short,
        }

        save_paths = None
        if save_artifacts:
            save_paths = save_model_artifacts(
                models=models_dict,
                metrics=df_metrics,
                probabilities=df_probabilities,
                output_dir=output_dir,
                experiment_name=experiment_name,
                metadata=metadata,
                overwrite=True,
                verbose=verbose,
            )

        # ==================================================
        # 4) Resumen final
        # ==================================================
        if verbose:
            print(
                f"\n[DONE] L{size} | "
                f"models={len(models_dict)} | "
                f"metrics_rows={len(df_metrics)} | "
                f"probabilities_rows={len(df_probabilities)}"
            )

        return {
            "models": models_dict,
            "metrics": df_metrics,
            "probabilities": df_probabilities,
            "metadata": metadata,
            "artifact_dir": save_paths["artifact_dir"] if save_paths is not None else None,
            "save_paths": save_paths,
        }

    finally:
        del bundles, results
        gc.collect()

## **10.4. Ejecución de entrenamientos**

### **Carga de hiperparámetros**

In [66]:
lr_model_params = lr_config["model_params"]
lr_rule_balanced = lr_config["decision_rules"]["balanced"]
lr_rule_conservative = lr_config["decision_rules"]["conservative"]
lr_rule_baseline = lr_config["decision_rules"]["baseline"]

### **LR baseline**

In [69]:
results_lr_baseline = run_logistic(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    experiment_name="lr_final_baseline",
    output_dir=OUTPUT_DIR_BEST_MODEL,
    C=lr_model_params["C"],
    max_iter=lr_model_params["max_iter"],
    solver=lr_model_params["solver"],
    multi_class=lr_model_params["multi_class"],
    class_weight=lr_model_params["class_weight"],
    random_state=lr_model_params["random_state"],
    input_mode=lr_model_params["input_mode"],
    prob_threshold_long=lr_rule_baseline["threshold_long"],
    prob_threshold_short=lr_rule_baseline["threshold_short"],
    verbose=True,
)

2026-04-23 19:27:42,689 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 19:27:42,690 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 19:27:42,715 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 19:27:42,716 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 19:27:42,737 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 19:27:42,737 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 19:27:42,742 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 19:27:42,743 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)



LOGISTIC REGRESSION | T2 SEQ2ONE | WINDOW_SIZE=L30
experiment_name = lr_final_baseline
targets         = ['t2_p40_h30', 't2_p50_h30']
class_weight    = balanced
input_mode      = 2d_flat
C               = 0.01
max_iter        = 1000
solver          = lbfgs
multi_class     = multinomial
random_state    = 42
thr_long        = 0.4
thr_short       = 0.4

[BUILD] L30 | n_targets=2


2026-04-23 19:27:42,830 | INFO | Loaded: windows_t2_p50_h30_train.npz
2026-04-23 19:27:42,831 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 19:27:42,853 | INFO | Loaded: windows_t2_p50_h30_valid.npz
2026-04-23 19:27:42,854 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 19:27:42,877 | INFO | Loaded: windows_t2_p50_h30_test.npz
2026-04-23 19:27:42,878 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 19:27:42,882 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 19:27:42,883 | INFO | Bundle cargado | target=t2_p50_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)



WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=logistic_regression | class_weight=balanced
  -> L30 | target=t2_p40_h30 | model=logistic_regression | class_weight=balanced
  -> L30 | target=t2_p50_h30 | model=logistic_regression | class_weight=balanced

ARTEFACTOS GUARDADOS
artifact_dir   = /content/drive/MyDrive/neural_profit/best_models/lr_final_baseline
n_models       = 2
metrics_saved  = True
prob_saved     = True
metadata_saved = True

[DONE] L30 | models=2 | metrics_rows=2 | probabilities_rows=13764


### **LR balanced**

In [67]:
results_lr_balanced = run_logistic(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    experiment_name="lr_final_balanced",
    output_dir=OUTPUT_DIR_BEST_MODEL,
    C=lr_model_params["C"],
    max_iter=lr_model_params["max_iter"],
    solver=lr_model_params["solver"],
    multi_class=lr_model_params["multi_class"],
    class_weight=lr_model_params["class_weight"],
    random_state=lr_model_params["random_state"],
    input_mode=lr_model_params["input_mode"],
    prob_threshold_long=lr_rule_balanced["threshold_long"],
    prob_threshold_short=lr_rule_balanced["threshold_short"],
    verbose=True,
)


EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS
experiment_name = lr_final_balanced
artifact_dir    = /content/drive/MyDrive/neural_profit/best_models/lr_final_balanced

ARTEFACTOS CARGADOS
artifact_dir = /content/drive/MyDrive/neural_profit/best_models/lr_final_balanced
n_models     = 2
metrics_rows = 2
prob_rows    = 13764


### **LR conservative**

In [68]:
results_lr_conservative = run_logistic(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    experiment_name="lr_final_conservative",
    output_dir=OUTPUT_DIR_BEST_MODEL,
    C=lr_model_params["C"],
    max_iter=lr_model_params["max_iter"],
    solver=lr_model_params["solver"],
    multi_class=lr_model_params["multi_class"],
    class_weight=lr_model_params["class_weight"],
    random_state=lr_model_params["random_state"],
    input_mode=lr_model_params["input_mode"],
    prob_threshold_long=lr_rule_conservative["threshold_long"],
    prob_threshold_short=lr_rule_conservative["threshold_short"],
    verbose=True,
)


EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS
experiment_name = lr_final_conservative
artifact_dir    = /content/drive/MyDrive/neural_profit/best_models/lr_final_conservative

ARTEFACTOS CARGADOS
artifact_dir = /content/drive/MyDrive/neural_profit/best_models/lr_final_conservative
n_models     = 2
metrics_rows = 2
prob_rows    = 13764


# **11. Modelo GRU**

## **Hiperparámetros seleccionados**

In [ ]:
# =========================================================
# GRU | Configuración final de hiperparámetros
# =========================================================

gru_config = {
    "model_name": "gru",

    # -----------------------------------------------------
    # Configuraciones de modelo (arquitectura + entrenamiento)
    # -----------------------------------------------------
    "model_params": {

        # Configuración base (benchmark predictivo)
        "baseline": {
            "hidden_size": 128,
            "num_layers": 2,
            "learning_rate": 5e-4,
            "dropout": 0.0,
            "batch_size": 2048,
            "grad_clip_norm": 0.5,
        },

        # Configuración operativa (alta precisión)
        "conservative": {
            "hidden_size": 256,
            "num_layers": 1,
            "learning_rate": 1e-3,
            "dropout": 0.25,  # valor intermedio dentro de 0.2–0.3
            "batch_size": 4096,
            "grad_clip_norm": 0.5,
        },
    },

    # -----------------------------------------------------
    # Reglas de decisión (thresholds)
    # -----------------------------------------------------
    "decision_rules": {

        # Benchmark (alineado con evaluación clásica)
        "baseline": {
            "threshold_long": 0.40,
            "threshold_short": 0.40,
        },

        # Configuración operativa (alta precisión)
        "conservative": {
            "threshold_long": 0.45,
            "threshold_short": 0.45,
        },
    },
}

## **10.1. Función unitaria por bundle**

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn

class GRUClassifier(nn.Module):
    def __init__(
        self,
        n_features: int,
        hidden_size: int = 64,
        num_layers: int = 1,
        dropout: float = 0.0,
        num_classes: int = 3,
    ):
        super().__init__()

        gru_dropout = dropout if num_layers > 1 else 0.0

        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=gru_dropout,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.gru(x)          # (batch, seq_len, hidden_size)
        last_out = out[:, -1, :]      # many-to-one
        last_out = self.dropout(last_out)
        logits = self.fc(last_out)    # (batch, num_classes)
        return logits


def run_gru_final_seq2one(
    bundle,
    *,
    random_state: int = 42,
    device: str | None = None,
    deterministic: bool = True,
    num_workers: int = 0,
    verbose: bool = False,
):
    """
    GRU FINAL (configuración óptima fija)

    - window_size = 30
    - target = t2_dir_thr_90
    - Sin tuning (parámetros cerrados)
    """

    # =========================
    # CONFIG FINAL (FIJA)
    # =========================
    hidden_size = 128
    num_layers = 1
    dropout = 0.1

    learning_rate = 1e-3
    weight_decay = 0.0

    batch_size = 2048
    eval_batch_size = 2048

    epochs = 20
    patience = 5

    class_weight = "balanced"
    grad_clip_norm = 1.0  # <-- NUEVO

    # =========================
    # 1. SEEDS Y DEVICE
    # =========================
    torch.manual_seed(random_state)
    np.random.seed(random_state)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    use_pin_memory = device == "cuda"

    # =========================
    # 2. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    X_test = bundle["test"]["X"]

    # =========================
    # VALIDACIÓN WINDOW SIZE
    # =========================
    if X_train.shape[1] != 30:
        raise ValueError(f"Se esperaba window_size=30, recibido {X_train.shape[1]}")

    # =========================
    # 3. SHAPES
    # =========================
    n_features = X_train.shape[2]

    # =========================
    # 4. LABEL ENCODING
    # =========================
    classes_ = np.sort(np.unique(y_train))
    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int64)
    y_valid_enc = np.array([class_to_idx[y] for y in y_valid], dtype=np.int64)

    num_classes = len(classes_)

    # =========================
    # 5. CLASS WEIGHTS (BALANCED)
    # =========================
    counts = np.bincount(y_train_enc, minlength=num_classes)
    total = counts.sum()

    weights = total / (num_classes * counts)

    criterion_weight = torch.tensor(
        weights, dtype=torch.float32, device=device
    )

    # =========================
    # 6. TENSORES
    # =========================
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_enc, dtype=torch.long)

    X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
    y_valid_t = torch.tensor(y_valid_enc, dtype=torch.long)

    X_test_t = torch.tensor(X_test, dtype=torch.float32)

    # =========================
    # 7. DATALOADERS
    # =========================
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train_t, y_train_t),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    valid_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_valid_t, y_valid_t),
        batch_size=eval_batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    test_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_test_t),
        batch_size=eval_batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    # =========================
    # 8. MODELO
    # =========================
    model = GRUClassifier(
        n_features=n_features,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        num_classes=num_classes,
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=criterion_weight)

    optimizer = torch.optim.AdamW(   # <-- CAMBIO IMPORTANTE
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    # =========================
    # HELPERS
    # =========================
    def _move(x):
        return x.to(device, non_blocking=True) if device == "cuda" else x.to(device)

    def compute_valid_loss():
        model.eval()
        total_loss = 0
        n = 0

        with torch.no_grad():
            for xb, yb in valid_loader:
                xb, yb = _move(xb), _move(yb)
                logits = model(xb)
                loss = criterion(logits, yb)

                total_loss += loss.item() * xb.size(0)
                n += xb.size(0)

        return total_loss / n

    def predict(loader):
        model.eval()
        out = []

        with torch.no_grad():
            for batch in loader:
                xb = _move(batch[0])
                logits = model(xb)
                out.append(logits.cpu())

        return torch.cat(out, dim=0)

    # =========================
    # 9. TRAIN
    # =========================
    best_state = None
    best_loss = np.inf
    wait = 0

    for epoch in range(epochs):
        model.train()

        for xb, yb in train_loader:
            xb, yb = _move(xb), _move(yb)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()

            # ===== CLIP GRADIENT =====
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            optimizer.step()

        val_loss = compute_valid_loss()

        if verbose:
            print(f"[Epoch {epoch+1}] valid_loss={val_loss:.6f}")

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)

    # =========================
    # 10. PREDICT
    # =========================
    valid_logits = predict(valid_loader)
    test_logits = predict(test_loader)

    y_pred_valid = valid_logits.argmax(1).numpy()
    y_pred_test = test_logits.argmax(1).numpy()

    y_pred_valid = np.array([idx_to_class[i] for i in y_pred_valid])
    y_pred_test = np.array([idx_to_class[i] for i in y_pred_test])

    return {
        "model": model,
        "y_pred_valid": y_pred_valid,
        "y_pred_test": y_pred_test,
        "best_valid_loss": best_loss,
    }

## **10.2. Función de evaluación sobre uno o más bundles**

In [ ]:
def eval_gru_bundles_final(
    bundles,
    *,
    split: str = "valid",
    model_name: str = "gru_final",
    random_state: int = 42,
    device: str | None = None,
    num_workers: int = 0,
    verbose: bool = False,
):
    """
    Evalúa GRU FINAL (configuración óptima fija)

    - window_size = 30
    - target = t2_dir_thr_90
    - split = "valid" o "test"
    """

    import gc
    import pandas as pd
    import torch

    if split not in {"valid", "test"}:
        raise ValueError("split debe ser 'valid' o 'test'")

    # -------------------------------
    # Normalizar entrada
    # -------------------------------
    if isinstance(bundles, dict):
        bundles_list = [bundles]
    else:
        bundles_list = list(bundles)

    rows = []

    # -------------------------------
    # Loop bundles
    # -------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"split={split} | "
                f"model={model_name}"
            )

        preds = None

        try:
            # -------------------------------
            # 1) RUN GRU FINAL
            # -------------------------------
            preds = run_gru_final_seq2one(
                bundle,
                random_state=random_state,
                device=device,
                num_workers=num_workers,
                verbose=False,
            )

            # -------------------------------
            # 2) SELECCIÓN DEL SPLIT
            # -------------------------------
            y_true = bundle[split]["y"]

            y_pred_key = f"y_pred_{split}"
            if y_pred_key not in preds:
                raise KeyError(f"No existe '{y_pred_key}' en preds")

            y_pred = preds[y_pred_key]

            # -------------------------------
            # 3) MÉTRICAS
            # -------------------------------
            metrics = compute_classification_metrics(
                y_true=y_true,
                y_pred=y_pred,
                model_name=model_name,
                split=split,
                target=target,
                labels=[-1, 0, 1],
            )

            # -------------------------------
            # 4) DATAFRAME
            # -------------------------------
            df_row = metrics_to_df(
                metrics,
                model=model_name,
                split=split,
                window_size=window_size,
                target=target,
            )

            df_row["horizon_min"] = horizon
            df_row["class_weight_mode"] = "balanced"

            rows.append(df_row)

        finally:
            # -------------------------------
            # CLEAN MEMORY
            # -------------------------------
            if preds is not None:
                del preds

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # -------------------------------
    # OUTPUT
    # -------------------------------
    return pd.concat(rows, ignore_index=True)

## **10.3. Función orquestadora por `window_size`**

In [ ]:
import gc
import pandas as pd
import torch


def run_gru(
    window_size: int,
    *,
    split: str = "valid",
    verbose: bool = True,
    model_name: str = "gru_final",
    random_state: int = 42,
    device: str | None = None,
    num_workers: int = 0,
) -> pd.DataFrame:
    """
    Ejecuta el GRU final para una sola configuración cerrada:

    - window_size = 30
    - target = t2_dir_thr_90
    - split = "valid" o "test"

    La arquitectura e hiperparámetros ya están fijados en
    run_gru_final_seq2one().
    """

    size = int(window_size)

    if split not in {"valid", "test"}:
        raise ValueError("split debe ser 'valid' o 'test'")

    if size != 30:
        raise ValueError(
            f"run_gru final solo admite window_size=30. Recibido: {size}"
        )

    bundle_t2_90 = None
    bundles_t2 = None
    df_out = None

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"GRU FINAL | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print("target = t2_dir_thr_90")
            print(f"split = {split}")
            print("hidden_size = 128")
            print("num_layers = 1")
            print("dropout = 0.1")
            print("learning_rate = 1e-3")
            print("batch_size = 2048")
            print("grad_clip_norm = 1.0")
            print("optimizer = AdamW")
            print("class_weight = balanced")

        # --------------------------------------------------
        # 2) Construcción del bundle
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | targets = ['t2_dir_thr_90']")

        bundle_t2_90, _ = create_bundles(
            window_size=size,
            targets=["t2_dir_thr_90", "t2_dir_thr_120"],
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        bundles_t2 = [bundle_t2_90]

        # --------------------------------------------------
        # 3) Evaluación
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[EVAL] L{size} | split={split} | "
                f"model={model_name} | target=t2_dir_thr_90"
            )

        df_out = eval_gru_bundles_final(
            bundles_t2,
            split=split,
            model_name=model_name,
            random_state=random_state,
            device=device,
            num_workers=num_workers,
            verbose=verbose,
        )

        # --------------------------------------------------
        # 4) Consolidación final
        # --------------------------------------------------
        df_out = (
            df_out
            .sort_values(["window_size", "target", "split", "horizon_min", "model"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 5) Resumen final
        # --------------------------------------------------
        if verbose:
            print(f"\n[DONE] L{size} | rows={len(df_out)}")
            print(
                df_out[
                    [
                        "window_size",
                        "split",
                        "target",
                        "model",
                        "horizon_min",
                        "class_weight_mode",
                        "balanced_accuracy",
                        "f1_macro",
                    ]
                ]
                .sort_values(
                    ["split", "target", "model", "horizon_min", "class_weight_mode"]
                )
                .to_string(index=False)
            )

        return df_out

    finally:
        # --------------------------------------------------
        # 6) Liberación de memoria
        # --------------------------------------------------
        del bundle_t2_90, bundles_t2
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## **10.4. Función incremental multi-ventana**

In [ ]:
from pathlib import Path
import gc
import pandas as pd
import torch


def run_gru_incremental(
    *,
    window_sizes: list[int],
    name: str = "gru_final",
    split: str = "valid",
    verbose: bool = True,
    random_state: int = 42,
    device: str | None = None,
    num_workers: int = 0,
) -> pd.DataFrame:
    """
    Ejecuta GRU FINAL de forma incremental.

    Configuración cerrada:
    - window_size = 30
    - target = t2_dir_thr_90
    - split = "valid" o "test"
    - class_weight = balanced
    """

    if split not in {"valid", "test"}:
        raise ValueError("split debe ser 'valid' o 'test'")

    metrics_dir = DRIVE_DIR / "metrics/post_tuning_metrics"
    metrics_path = metrics_dir / f"classification_{name}_metrics.parquet"

    # --------------------------------------------------
    # 1) Cargar histórico si existe
    # --------------------------------------------------
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    # --------------------------------------------------
    # 2) Definir completitud esperada
    # --------------------------------------------------
    expected_targets = {"t2_dir_thr_90"}
    expected_splits = {split}
    expected_models = {name}
    expected_class_weight_mode = {"balanced"}

    # --------------------------------------------------
    # 3) Iterar por window_sizes
    # --------------------------------------------------
    for L in window_sizes:
        L = int(L)
        dfL = None

        if L != 30:
            if verbose:
                print(f"[SKIP] {name} L={L} ignorado: el modelo final solo usa L=30")
            continue

        # ----------------------------------------------
        # Skip robusto por window_size
        # ----------------------------------------------
        if not df_hist.empty:
            dfL = df_hist[
                (df_hist["window_size"] == L) &
                (df_hist["split"] == split)
            ]

            done_targets = set(dfL["target"].unique()) if not dfL.empty else set()
            done_splits = set(dfL["split"].unique()) if not dfL.empty else set()
            done_models = set(dfL["model"].unique()) if not dfL.empty else set()

            done_class_weight_mode = (
                set(dfL["class_weight_mode"].dropna().unique())
                if ("class_weight_mode" in dfL.columns and not dfL.empty)
                else set()
            )

            is_complete = (
                expected_targets.issubset(done_targets)
                and expected_splits.issubset(done_splits)
                and expected_models.issubset(done_models)
                and expected_class_weight_mode.issubset(done_class_weight_mode)
            )

            if is_complete:
                if verbose:
                    print(f"[SKIP] {name} L={L} ya existe completo en Drive")
                continue

        # ----------------------------------------------
        # Ejecutar GRU para este window_size
        # ----------------------------------------------
        if verbose:
            print("\n" + "-" * 80)
            print(f"[RUN] {name} | L={L}")
            print(f"target=t2_dir_thr_90 | split={split} | class_weight=balanced")
            print("-" * 80)

        df_L = run_gru(
            window_size=L,
            split=split,
            verbose=verbose,
            model_name=name,
            random_state=random_state,
            device=device,
            num_workers=num_workers,
        )

        df_L["family"] = name

        # ----------------------------------------------
        # Actualizar histórico
        # ----------------------------------------------
        if df_hist.empty:
            df_hist = df_L.copy()
        else:
            df_hist = pd.concat([df_hist, df_L], ignore_index=True)

        # ----------------------------------------------
        # Eliminar duplicados por seguridad
        # ----------------------------------------------
        subset_cols = [
            "window_size",
            "target",
            "split",
            "model",
            "horizon_min",
            "class_weight_mode",
        ]

        df_hist = (
            df_hist
            .drop_duplicates(subset=subset_cols, keep="last")
            .reset_index(drop=True)
        )

        # ----------------------------------------------
        # Guardar histórico actualizado
        # ----------------------------------------------
        save_classification_metrics(df_hist, name=name)

        # ----------------------------------------------
        # Liberación explícita de memoria entre L
        # ----------------------------------------------
        del df_L, dfL
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --------------------------------------------------
    # 4) Retorno final ordenado
    # --------------------------------------------------
    sort_cols = [
        "window_size",
        "target",
        "split",
        "horizon_min",
        "model",
        "class_weight_mode",
    ]

    return df_hist.sort_values(sort_cols).reset_index(drop=True)

## **10.5. Ejecución final del experimento**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

df_gru_valid = run_gru_incremental(
    window_sizes=[30],
    name="gru_final",
    split="valid",
    verbose=True,
    device=device,
)

[SKIP] gru_final L=30 ya existe completo en Drive


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

df_gru_test = run_gru_incremental(
    window_sizes=[30],
    name="gru_final_test",
    split="test",
    verbose=True,
    device=device,
)

[SKIP] gru_final_test L=30 ya existe completo en Drive


# **12. Modelo XGBoost**

En base a estos resultados, se define la siguiente configuración final del modelo:

* `n_estimators = 600`
* `max_depth = 3`
* `learning_rate = 0.01`
* `subsample = 1.0`
* `colsample_bytree = 0.8`
* `gamma = 0.0`
* `reg_alpha = 0.1`
* `reg_lambda = 1.0`

Desde el punto de vista predictivo, esta configuración presenta el mejor desempeño promedio en términos de `balanced_accuracy` y `f1_macro`.

El análisis de probabilidades muestra que la principal fuente de variación operativa del modelo proviene de los thresholds de decisión, que determinan el equilibrio entre precisión y volumen de señales.

Se identifican dos configuraciones operativas principales:

Configuración balanceada:

* `threshold_long = 0.45`
* `threshold_short = 0.40`

Esta opción ofrece un compromiso adecuado entre:

* precisión
* frecuencia de operación
* tasa de señales útiles

Configuración conservadora:

* `threshold_long = 0.50`
* `threshold_short = 0.50`

Esta configuración prioriza señales de alta confianza, reduciendo significativamente la cantidad de operaciones.

## 11.1.

In [ ]:
from xgboost import XGBClassifier
import numpy as np


def run_xgboost_for_bundle_seq2one(
    bundle,
    *,
    n_estimators: int = 200,
    max_depth: int = 3,
    learning_rate: float = 0.03,
    subsample: float = 0.8,
    colsample_bytree: float = 0.8,
    min_child_weight: int = 1,
    gamma: float = 0.0,
    reg_alpha: float = 0.0,
    reg_lambda: float = 10.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight: str | dict | None = "balanced",
    use_gpu: bool = True,
    verbose: bool = False,
):
    """
    XGBoost final para clasificación T2.

    Configuración objetivo:
    - window_size = 30
    - target = t2_dir_thr_90

    Entrenamiento:
    - TRAIN -> fit

    Predicción:
    - VALID
    - TEST
    """

    # =========================
    # 1. DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    X_test = bundle["test"]["X"]
    y_test = bundle["test"]["y"]

    # =========================
    # 2. VALIDACIONES
    # =========================
    window_size = int(bundle.get("window_size", X_train.shape[1]))
    target = bundle.get("target", None)

    if window_size != 30:
        raise ValueError(
            f"Esta función final espera window_size=30. Recibido: {window_size}"
        )

    if target is not None and target != "t2_dir_thr_90":
        raise ValueError(
            f"Esta función final espera target='t2_dir_thr_90'. Recibido: {target}"
        )

    # =========================
    # 3. INPUT
    # =========================
    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)
    X_test_model = prepare_X_for_model(X_test, input_mode=input_mode)

    # =========================
    # 4. ENCODE LABELS
    # =========================
    classes_ = np.sort(np.unique(y_train))

    unknown_valid = set(np.unique(y_valid)) - set(classes_)
    if unknown_valid:
        raise ValueError(
            f"VALID contiene clases no vistas en TRAIN: {sorted(unknown_valid)}"
        )

    unknown_test = set(np.unique(y_test)) - set(classes_)
    if unknown_test:
        raise ValueError(
            f"TEST contiene clases no vistas en TRAIN: {sorted(unknown_test)}"
        )

    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int32)
    num_class = len(classes_)

    # =========================
    # 5. SAMPLE WEIGHT
    # =========================
    sample_weight = None
    weights_by_idx = None

    if class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_class)
        total = counts.sum()

        if np.any(counts == 0):
            raise ValueError(
                f"Hay clases sin muestras en TRAIN. counts={counts.tolist()}"
            )

        weights_by_idx = {
            idx: total / (num_class * count)
            for idx, count in enumerate(counts)
        }

        sample_weight = np.array(
            [weights_by_idx[idx] for idx in y_train_enc],
            dtype=np.float32,
        )

    elif isinstance(class_weight, dict):
        weights_by_idx = {
            class_to_idx[cls]: weight
            for cls, weight in class_weight.items()
            if cls in class_to_idx
        }

        sample_weight = np.array(
            [weights_by_idx.get(idx, 1.0) for idx in y_train_enc],
            dtype=np.float32,
        )

    elif class_weight is None:
        sample_weight = None

    else:
        raise ValueError("class_weight debe ser None, 'balanced' o dict")

    # =========================
    # 6. CONFIG BACKEND
    # =========================
    device = "cuda" if use_gpu else "cpu"

    # =========================
    # 7. MODELO
    # =========================
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=num_class,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        tree_method="hist",
        device=device,
        random_state=random_state,
        n_jobs=n_jobs,
        eval_metric="mlogloss",
        verbosity=1 if verbose else 0,
    )

    # =========================
    # 8. TRAIN
    # =========================
    model.fit(
        X_train_model,
        y_train_enc,
        sample_weight=sample_weight,
    )

    # =========================
    # 9. PREDICT VALID
    # =========================
    y_pred_valid_enc = model.predict(X_valid_model)
    y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])
    y_proba_valid = model.predict_proba(X_valid_model)

    # =========================
    # 10. PREDICT TEST
    # =========================
    y_pred_test_enc = model.predict(X_test_model)
    y_pred_test = np.array([idx_to_class[int(y)] for y in y_pred_test_enc])
    y_proba_test = model.predict_proba(X_test_model)

    return {
        "model": model,
        "classes_": classes_,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "weights_by_idx": weights_by_idx,
        "y_true_valid": y_valid,
        "y_pred_valid": y_pred_valid,
        "y_proba_valid": y_proba_valid,
        "y_true_test": y_test,
        "y_pred_test": y_pred_test,
        "y_proba_test": y_proba_test,
        "params": {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "min_child_weight": min_child_weight,
            "gamma": gamma,
            "reg_alpha": reg_alpha,
            "reg_lambda": reg_lambda,
            "class_weight": class_weight,
            "input_mode": input_mode,
            "device": device,
        },
    }

## 11.2.

In [ ]:
from typing import Any, Dict, List, Sequence, Union
import gc
import pandas as pd


def eval_xgboost_bundles_final(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    split: str = "valid",
    model_name: str = "xgboost_final",
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    use_gpu: bool = True,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Evalúa XGBoost FINAL (configuración óptima fija)

    - window_size = 30
    - target = t2_dir_thr_90
    - split = "valid" o "test"

    Configuración cerrada:
    - n_estimators = 200
    - max_depth = 3
    - learning_rate = 0.03
    - subsample = 0.8
    - colsample_bytree = 0.8
    - min_child_weight = 1
    - gamma = 0.0
    - reg_alpha = 0.0
    - reg_lambda = 10.0
    - class_weight = balanced
    """

    if split not in {"valid", "test"}:
        raise ValueError("split debe ser 'valid' o 'test'")

    # -------------------------------
    # Normalizar entrada
    # -------------------------------
    if isinstance(bundles, dict):
        bundles_list: List[Dict[str, Any]] = [bundles]
    else:
        bundles_list = list(bundles)

    rows = []

    # -------------------------------
    # Hiperparámetros finales
    # -------------------------------
    n_estimators = 200
    max_depth = 3
    learning_rate = 0.03
    subsample = 0.8
    colsample_bytree = 0.8
    min_child_weight = 1
    gamma = 0.0
    reg_alpha = 0.0
    reg_lambda = 10.0
    class_weight = "balanced"

    # -------------------------------
    # Loop bundles
    # -------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"split={split} | "
                f"model={model_name}"
            )

        preds = None

        try:
            # -------------------------------
            # 1) RUN XGBOOST FINAL
            # -------------------------------
            preds = run_xgboost_for_bundle_seq2one(
                bundle,
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                min_child_weight=min_child_weight,
                gamma=gamma,
                reg_alpha=reg_alpha,
                reg_lambda=reg_lambda,
                random_state=random_state,
                n_jobs=n_jobs,
                input_mode=input_mode,
                class_weight=class_weight,
                use_gpu=use_gpu,
                verbose=False,
            )

            # -------------------------------
            # 2) SELECCIÓN DEL SPLIT
            # -------------------------------
            y_true_key = f"y_true_{split}"
            y_pred_key = f"y_pred_{split}"

            if y_true_key not in preds:
                raise KeyError(f"No existe '{y_true_key}' en preds")
            if y_pred_key not in preds:
                raise KeyError(f"No existe '{y_pred_key}' en preds")

            y_true = preds[y_true_key]
            y_pred = preds[y_pred_key]

            # -------------------------------
            # 3) MÉTRICAS
            # -------------------------------
            metrics = compute_classification_metrics(
                y_true=y_true,
                y_pred=y_pred,
                model_name=model_name,
                split=split,
                target=target,
                labels=[-1, 0, 1],
            )

            # -------------------------------
            # 4) DATAFRAME
            # -------------------------------
            df_row = metrics_to_df(
                metrics,
                model=model_name,
                split=split,
                window_size=window_size,
                target=target,
            )

            df_row["horizon_min"] = horizon
            df_row["class_weight_mode"] = class_weight

            # Hiperparámetros finales
            df_row["n_estimators"] = n_estimators
            df_row["max_depth"] = max_depth
            df_row["learning_rate"] = learning_rate
            df_row["subsample"] = subsample
            df_row["colsample_bytree"] = colsample_bytree
            df_row["min_child_weight"] = min_child_weight
            df_row["gamma"] = gamma
            df_row["reg_alpha"] = reg_alpha
            df_row["reg_lambda"] = reg_lambda

            rows.append(df_row)

        finally:
            # -------------------------------
            # CLEAN MEMORY
            # -------------------------------
            if preds is not None:
                del preds

            gc.collect()

    # -------------------------------
    # OUTPUT
    # -------------------------------
    return pd.concat(rows, ignore_index=True)

## 11.3.

In [ ]:
import gc
import pandas as pd


def run_xgboost(
    window_size: int,
    *,
    split: str = "valid",
    verbose: bool = True,
    model_name: str = "xgboost_final",
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    use_gpu: bool = True,
) -> pd.DataFrame:
    """
    Ejecuta XGBoost FINAL para una sola configuración cerrada:

    - window_size = 30
    - target = t2_dir_thr_90
    - split = "valid" o "test"

    La configuración final queda fijada dentro de esta función.
    """

    size = int(window_size)

    if split not in {"valid", "test"}:
        raise ValueError("split debe ser 'valid' o 'test'")

    if size != 30:
        raise ValueError(
            f"run_xgboost final solo admite window_size=30. Recibido: {size}"
        )

    # --------------------------------------------------
    # Hiperparámetros finales
    # --------------------------------------------------
    n_estimators = 200
    max_depth = 3
    learning_rate = 0.03
    subsample = 0.8
    colsample_bytree = 0.8
    min_child_weight = 1
    gamma = 0.0
    reg_alpha = 0.0
    reg_lambda = 10.0
    class_weight = "balanced"

    bundle_t2_90 = None
    bundles_t2 = None
    df_out = None

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"XGBOOST FINAL | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print("target = t2_dir_thr_90")
            print(f"split = {split}")
            print(f"n_estimators     = {n_estimators}")
            print(f"max_depth        = {max_depth}")
            print(f"learning_rate    = {learning_rate}")
            print(f"subsample        = {subsample}")
            print(f"colsample_bytree = {colsample_bytree}")
            print(f"min_child_weight = {min_child_weight}")
            print(f"gamma            = {gamma}")
            print(f"reg_alpha        = {reg_alpha}")
            print(f"reg_lambda       = {reg_lambda}")
            print(f"class_weight     = {class_weight}")

        # --------------------------------------------------
        # 2) Construcción del bundle
        # create_bundles exige 2 targets, así que cargamos ambos
        # y luego usamos solo t2_dir_thr_90
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | targets = ['t2_dir_thr_90']")

        bundle_t2_90, _ = create_bundles(
            window_size=size,
            targets=["t2_dir_thr_90", "t2_dir_thr_120"],
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        bundles_t2 = [bundle_t2_90]

        # --------------------------------------------------
        # 3) Evaluación
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[EVAL] L{size} | split={split} | "
                f"model={model_name} | target=t2_dir_thr_90"
            )

        df_out = eval_xgboost_bundles_final(
            bundles_t2,
            split=split,
            model_name=model_name,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            use_gpu=use_gpu,
            verbose=verbose,
        )

        # --------------------------------------------------
        # 4) Orden final
        # --------------------------------------------------
        df_out = (
            df_out
            .sort_values(["window_size", "target", "split", "horizon_min", "model"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 5) Resumen
        # --------------------------------------------------
        if verbose:
            print(f"\n[DONE] L{size} | rows={len(df_out)}")
            print(
                df_out[
                    [
                        "window_size",
                        "split",
                        "target",
                        "model",
                        "horizon_min",
                        "class_weight_mode",
                        "n_estimators",
                        "max_depth",
                        "learning_rate",
                        "subsample",
                        "colsample_bytree",
                        "min_child_weight",
                        "gamma",
                        "reg_alpha",
                        "reg_lambda",
                        "balanced_accuracy",
                        "f1_macro",
                    ]
                ].to_string(index=False)
            )

        return df_out

    finally:
        # --------------------------------------------------
        # 6) Liberación de memoria
        # --------------------------------------------------
        del bundle_t2_90, bundles_t2
        gc.collect()

## 12.4.

In [ ]:
from pathlib import Path
import gc
import pandas as pd


def run_xgboost_incremental(
    *,
    window_sizes: list[int],
    name: str = "xgboost_final",
    split: str = "valid",
    verbose: bool = True,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    use_gpu: bool = True,
) -> pd.DataFrame:
    """
    Ejecuta XGBoost FINAL de forma incremental.

    Configuración cerrada:
    - window_size = 30
    - target = t2_dir_thr_90
    - split = "valid" o "test"
    - class_weight = balanced
    """

    if split not in {"valid", "test"}:
        raise ValueError("split debe ser 'valid' o 'test'")

    metrics_dir = DRIVE_DIR / "metrics/post_tuning_metrics"
    metrics_path = metrics_dir / f"classification_{name}_metrics.parquet"

    # --------------------------------------------------
    # 1) Cargar histórico
    # --------------------------------------------------
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    # --------------------------------------------------
    # 2) Config final esperada
    # --------------------------------------------------
    n_estimators = 200
    max_depth = 3
    learning_rate = 0.03
    subsample = 0.8
    colsample_bytree = 0.8
    min_child_weight = 1
    gamma = 0.0
    reg_alpha = 0.0
    reg_lambda = 10.0
    class_weight = "balanced"

    expected_combos = {
        ("t2_dir_thr_90", split),
    }

    # --------------------------------------------------
    # 3) Loop por window_sizes
    # --------------------------------------------------
    for L in window_sizes:
        L = int(L)
        df_existing = None

        # El modelo final solo admite L=30
        if L != 30:
            if verbose:
                print(f"[SKIP] {name} L={L} ignorado: el modelo final solo usa L=30")
            continue

        # ----------------------------------------------
        # 3.1) Filtrar histórico para esta config exacta
        # ----------------------------------------------
        if not df_hist.empty:
            mask = (
                (df_hist["window_size"] == L)
                & (df_hist["split"] == split)
                & (df_hist["model"] == name)
                & (df_hist["class_weight_mode"] == class_weight)
                & (df_hist["n_estimators"] == n_estimators)
                & (df_hist["max_depth"] == max_depth)
                & (df_hist["learning_rate"] == learning_rate)
                & (df_hist["subsample"] == subsample)
                & (df_hist["colsample_bytree"] == colsample_bytree)
                & (df_hist["min_child_weight"] == min_child_weight)
                & (df_hist["gamma"] == gamma)
                & (df_hist["reg_alpha"] == reg_alpha)
                & (df_hist["reg_lambda"] == reg_lambda)
            )

            df_existing = df_hist.loc[mask].copy()

            if not df_existing.empty:
                combos_done = set(zip(df_existing["target"], df_existing["split"]))
                is_complete = expected_combos.issubset(combos_done)
            else:
                is_complete = False

            if is_complete:
                if verbose:
                    print(f"[SKIP] {name} | L={L} | configuración final ya existe")
                continue

        # ----------------------------------------------
        # 3.2) Ejecutar
        # ----------------------------------------------
        if verbose:
            print("\n" + "-" * 100)
            print(f"[RUN] {name} | L={L}")
            print(f"target=t2_dir_thr_90 | split={split} | class_weight=balanced")
            print(
                f"n_estimators={n_estimators} | "
                f"max_depth={max_depth} | "
                f"learning_rate={learning_rate} | "
                f"subsample={subsample} | "
                f"colsample_bytree={colsample_bytree} | "
                f"min_child_weight={min_child_weight} | "
                f"gamma={gamma} | "
                f"reg_alpha={reg_alpha} | "
                f"reg_lambda={reg_lambda}"
            )
            print("-" * 100)

        df_L = run_xgboost(
            window_size=L,
            split=split,
            verbose=verbose,
            model_name=name,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            use_gpu=use_gpu,
        )

        # familia
        df_L["family"] = name

        # ----------------------------------------------
        # 3.3) Actualizar histórico
        # ----------------------------------------------
        if df_hist.empty:
            df_hist = df_L.copy()
        else:
            df_hist = pd.concat([df_hist, df_L], ignore_index=True)

        # ----------------------------------------------
        # 3.4) Drop duplicates
        # ----------------------------------------------
        subset_cols = [
            "window_size",
            "target",
            "split",
            "model",
            "horizon_min",
            "class_weight_mode",
            "n_estimators",
            "max_depth",
            "learning_rate",
            "subsample",
            "colsample_bytree",
            "min_child_weight",
            "gamma",
            "reg_alpha",
            "reg_lambda",
        ]

        df_hist = (
            df_hist
            .drop_duplicates(subset=subset_cols, keep="last")
            .reset_index(drop=True)
        )

        # ----------------------------------------------
        # 3.5) Guardar
        # ----------------------------------------------
        save_classification_metrics(df_hist, name=name)

        # ----------------------------------------------
        # 3.6) Liberar memoria
        # ----------------------------------------------
        del df_L, df_existing
        gc.collect()

    # --------------------------------------------------
    # 4) Orden final
    # --------------------------------------------------
    sort_cols = [
        "window_size",
        "target",
        "split",
        "horizon_min",
        "model",
        "n_estimators",
        "max_depth",
        "learning_rate",
        "min_child_weight",
        "gamma",
        "reg_alpha",
        "reg_lambda",
    ]

    return df_hist.sort_values(sort_cols).reset_index(drop=True)

## 12.5.

In [ ]:
df_xgb_valid = run_xgboost_incremental(
    window_sizes=[30],
    name="xgboost_final",
    split="valid",
    verbose=True,
    use_gpu=True,
)

[SKIP] xgboost_final | L=30 | configuración final ya existe


In [ ]:
df_xgb_test = run_xgboost_incremental(
    window_sizes=[30],
    name="xgboost_final_test",
    split="test",
    verbose=True,
    use_gpu=True,
)

[SKIP] xgboost_final_test | L=30 | configuración final ya existe


# **12. Comparación**

In [ ]:
import pandas as pd

# ============================================================
# 1) Copias defensivas
# ============================================================
dfs = [
    df_gru_valid.copy(),
    df_gru_test.copy(),
    df_xgb_valid.copy(),
    df_xgb_test.copy(),
]

# ============================================================
# 2) Unión de columnas
# ============================================================
all_cols = sorted(set().union(*[df.columns for df in dfs]))

dfs_aligned = [df.reindex(columns=all_cols) for df in dfs]

# ============================================================
# 3) Concatenación
# ============================================================
df_all_models = pd.concat(dfs_aligned, ignore_index=True)

# ============================================================
# 4) Orden de columnas (enfocado en comparación)
# ============================================================
preferred_cols = [
    "model",
    "family",
    "split",
    "window_size",
    "target",
    "horizon_min",
    "n_samples",
    "class_weight_mode",
    "balanced_accuracy",
    "balanced_accuracy_naive",
    "bal_acc_gain_vs_naive",
    "f1_macro",
    "f1_weighted",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "min_child_weight",
    "gamma",
    "reg_alpha",
    "reg_lambda",
]

remaining_cols = [c for c in df_all_models.columns if c not in preferred_cols]

df_all_models = df_all_models[
    [c for c in preferred_cols if c in df_all_models.columns] + remaining_cols
]

# ============================================================
# 5) Orden de filas
# ============================================================
df_all_models = df_all_models.sort_values(
    ["split", "model"]
).reset_index(drop=True)

# ============================================================
# 6) Resultado final
# ============================================================
display(df_all_models)

,model,family,split,window_size,target,horizon_min,n_samples,class_weight_mode,balanced_accuracy,balanced_accuracy_naive,...,recall_macro,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda
0,gru_final_test,gru_final_test,test,30,t2_dir_thr_90,90,93990,balanced,0.432091,0.333333,...,0.432091,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,xgboost_final_test,xgboost_final_test,test,30,t2_dir_thr_90,90,93990,balanced,0.439816,0.333333,...,0.439816,200.0,3.0,0.03,0.8,0.8,1.0,0.0,0.0,10.0
2,gru_final,gru_final,valid,30,t2_dir_thr_90,90,93508,balanced,0.431711,0.333333,...,0.431711,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,xgboost_final,xgboost_final,valid,30,t2_dir_thr_90,90,93508,balanced,0.444060,0.333333,...,0.444060,200.0,3.0,0.03,0.8,0.8,1.0,0.0,0.0,10.0


In [ ]:
df_compare = df_all_models[
    [
        "model",
        "family",
        "split",
        "window_size",
        "target",
        "balanced_accuracy",
        "bal_acc_gain_vs_naive",
        "f1_macro",
        "accuracy",
    ]
].copy()

display(df_compare)

,model,family,split,window_size,target,balanced_accuracy,bal_acc_gain_vs_naive,f1_macro,accuracy
0,gru_final_test,gru_final_test,test,30,t2_dir_thr_90,0.432091,0.098758,0.430598,0.524673
1,xgboost_final_test,xgboost_final_test,test,30,t2_dir_thr_90,0.439816,0.106482,0.439437,0.536493
2,gru_final,gru_final,valid,30,t2_dir_thr_90,0.431711,0.098378,0.428348,0.634352
3,xgboost_final,xgboost_final,valid,30,t2_dir_thr_90,0.444060,0.110727,0.445724,0.647923


In [ ]:
df_metrics = df_all_models[
    [
        "model",
        "split",
        "balanced_accuracy",
        "balanced_accuracy_naive",
        "bal_acc_gain_vs_naive",
        "f1_macro",
        "f1_weighted",
        "accuracy",
        "precision_macro",
        "recall_macro",
    ]
].copy()

display(df_metrics)

,model,split,balanced_accuracy,balanced_accuracy_naive,bal_acc_gain_vs_naive,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro
0,gru_final_test,test,0.432091,0.333333,0.098758,0.430598,0.528047,0.524673,0.434409,0.432091
1,xgboost_final_test,test,0.439816,0.333333,0.106482,0.439437,0.535982,0.536493,0.439409,0.439816
2,gru_final,valid,0.431711,0.333333,0.098378,0.428348,0.633987,0.634352,0.430275,0.431711
3,xgboost_final,valid,0.444060,0.333333,0.110727,0.445724,0.644814,0.647923,0.447728,0.444060


## **Conclusión final — Valid vs Test**



1. Consistencia entre valid y test

Ambos modelos muestran una diferencia muy pequeña entre VALID y TEST:

- GRU:
  - valid: 0.4317
  - test: 0.4321

- XGBoost:
  - valid: 0.4441
  - test: 0.4398

La caída es mínima en XGBoost (~0.004) y prácticamente nula en GRU.

Esto indica que:

- no hay sobreajuste significativo
- la señal es estable
- el pipeline está bien construido

2. Comparación final entre modelos

XGBoost sigue superando a GRU tanto en VALID como en TEST:

- Balanced accuracy (test):
  - GRU: 0.4321
  - XGBoost: 0.4398

- Gain vs naive:
  - GRU: +0.0988
  - XGBoost: +0.1065

- F1 macro:
  - GRU: 0.4306
  - XGBoost: 0.4394

La ventaja de XGBoost se mantiene fuera de muestra.

3. Interpretación técnica

El hecho de que:

- VALID ≈ TEST
- y XGBoost > GRU en ambos

implica que:

- la selección de modelo fue correcta
- no hubo sesgo en validation
- la señal capturada es real y generaliza

Además:

- GRU no logra aprovechar mejor la estructura temporal
- XGBoost captura mejor la relación entre features

4. Nivel de señal

El nivel final de señal se mantiene en:

- XGBoost: ~0.106 de gain
- GRU: ~0.099 de gain

Esto confirma que:

- la señal existe
- pero es moderada
- no hay mejoras ocultas adicionales

5. Conclusión final del proyecto

- El target T2 contiene señal real y explotable
- El modelo XGBoost es la mejor solución encontrada
- El modelo generaliza correctamente a datos no vistos
- El pipeline completo (train → valid → test) es consistente

6. Conclusión operativa

El modelo está listo para el siguiente paso:

- construcción de reglas de trading
- evaluación de rentabilidad (backtest)

No se requieren más ajustes de modelado en esta etapa.

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 1) Cargar bundle (mismo dataset para ambos modelos)
# ============================================================
bundle_t2_90, _ = create_bundles(
    window_size=30,
    targets=["t2_dir_thr_90", "t2_dir_thr_120"],
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
)

bundle = bundle_t2_90

# ============================================================
# 2) Predicciones GRU y XGBoost sobre TEST
# ============================================================
preds_gru = run_gru_final_seq2one(bundle)
preds_xgb = run_xgboost_for_bundle_seq2one(bundle)

y_true = bundle["test"]["y"]
y_pred_gru = preds_gru["y_pred_test"]
y_pred_xgb = preds_xgb["y_pred_test"]

# ============================================================
# 3) Dataset base de comparación
# ============================================================
df_preds = pd.DataFrame({
    "y_true": y_true,
    "gru": y_pred_gru,
    "xgb": y_pred_xgb,
})

df_preds["agree"] = df_preds["gru"] == df_preds["xgb"]
df_preds["gru_correct"] = df_preds["gru"] == df_preds["y_true"]
df_preds["xgb_correct"] = df_preds["xgb"] == df_preds["y_true"]

# quién gana en cada fila
df_preds["winner"] = np.select(
    [
        df_preds["gru_correct"] & ~df_preds["xgb_correct"],
        ~df_preds["gru_correct"] & df_preds["xgb_correct"],
        df_preds["gru_correct"] & df_preds["xgb_correct"],
        ~df_preds["gru_correct"] & ~df_preds["xgb_correct"],
    ],
    [
        "gru_only",
        "xgb_only",
        "both_correct",
        "both_wrong",
    ],
    default="unknown",
)

# ============================================================
# 4) Resumen general
# ============================================================
summary_general = pd.DataFrame({
    "metric": [
        "n_samples",
        "gru_accuracy",
        "xgb_accuracy",
        "agreement_rate",
        "disagreement_rate",
    ],
    "value": [
        len(df_preds),
        df_preds["gru_correct"].mean(),
        df_preds["xgb_correct"].mean(),
        df_preds["agree"].mean(),
        1 - df_preds["agree"].mean(),
    ]
})

# ============================================================
# 5) Análisis de desacuerdos
# ============================================================
df_disagree = df_preds[df_preds["gru"] != df_preds["xgb"]].copy()

summary_disagree = pd.DataFrame({
    "metric": [
        "n_disagreements",
        "gru_accuracy_when_disagree",
        "xgb_accuracy_when_disagree",
        "gru_only_wins",
        "xgb_only_wins",
        "both_wrong_when_disagree",
    ],
    "value": [
        len(df_disagree),
        df_disagree["gru_correct"].mean() if len(df_disagree) > 0 else np.nan,
        df_disagree["xgb_correct"].mean() if len(df_disagree) > 0 else np.nan,
        (df_disagree["winner"] == "gru_only").mean() if len(df_disagree) > 0 else np.nan,
        (df_disagree["winner"] == "xgb_only").mean() if len(df_disagree) > 0 else np.nan,
        (df_disagree["winner"] == "both_wrong").mean() if len(df_disagree) > 0 else np.nan,
    ]
})

# ============================================================
# 6) Matriz de combinaciones GRU vs XGB
# ============================================================
combo_matrix = (
    df_preds
    .groupby(["gru", "xgb"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

# ============================================================
# 7) Precisión por coincidencia exacta entre modelos
# ============================================================
agreement_detail_rows = []

for cls in [-1, 0, 1]:
    df_cls = df_preds[(df_preds["gru"] == cls) & (df_preds["xgb"] == cls)].copy()
    agreement_detail_rows.append({
        "agreed_class": cls,
        "n_cases": len(df_cls),
        "share_of_total": len(df_cls) / len(df_preds),
        "precision_given_agreement": (df_cls["y_true"] == cls).mean() if len(df_cls) > 0 else np.nan,
    })

agreement_detail = pd.DataFrame(agreement_detail_rows)

# ============================================================
# 8) Tabla cruzada con verdad real en casos de acuerdo
# ============================================================
agree_only = df_preds[df_preds["agree"]].copy()
agree_vs_truth = (
    agree_only
    .groupby(["gru", "y_true"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

# ============================================================
# 9) Tabla cruzada con verdad real en casos de desacuerdo
# ============================================================
disagree_vs_truth = (
    df_disagree
    .groupby(["gru", "xgb", "y_true"])
    .size()
    .reset_index(name="count")
    .sort_values(["gru", "xgb", "y_true"])
    .reset_index(drop=True)
)

# ============================================================
# 10) Tabla corta de predicciones (primeras filas)
# ============================================================
preview_preds = df_preds.head(20).copy()

# ============================================================
# 11) Impresión ordenada
# ============================================================
print("=" * 90)
print("RESUMEN GENERAL")
print("=" * 90)
display(summary_general)

print("=" * 90)
print("RESUMEN DE DESACUERDOS")
print("=" * 90)
display(summary_disagree)

print("=" * 90)
print("MATRIZ DE COMBINACIONES: GRU vs XGBoost")
print("=" * 90)
display(combo_matrix)

print("=" * 90)
print("PRECISIÓN CUANDO AMBOS MODELOS COINCIDEN EN LA MISMA CLASE")
print("=" * 90)
display(agreement_detail)

print("=" * 90)
print("VERDAD REAL CUANDO AMBOS MODELOS COINCIDEN")
print("=" * 90)
display(agree_vs_truth)

print("=" * 90)
print("VERDAD REAL CUANDO LOS MODELOS DISCREPAN")
print("=" * 90)
display(disagree_vs_truth)

print("=" * 90)
print("MUESTRA DE PREDICCIONES")
print("=" * 90)
display(preview_preds)

2026-04-14 23:52:21,736 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-04-14 23:52:21,737 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 23:52:22,366 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-04-14 23:52:22,367 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 23:52:22,848 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-04-14 23:52:22,849 | INFO | X shape: (93990, 30, 5) | y shape: (93990,)
2026-04-14 23:52:23,198 | INFO | Scaler cargado: scaler_t2.pkl
2026-04-14 23:52:23,199 | INFO | Bundle cargado | target=t2_dir_thr_90 | window_size=30 | train=(436692, 30, 5) | valid=(93508, 30, 5) | test=(93990, 30, 5)
2026-04-14 23:52:24,903 | INFO | Loaded: windows_t2_dir_thr_120_train.npz
2026-04-14 23:52:24,904 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 23:52:25,510 | INFO | Loaded: windows_t2_dir_thr_120_valid.npz
2026-04-14 23:52:25,511 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 23:52:25,983 |


WINDOW_SIZE: 30

TARGET: t2_dir_thr_90
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

TARGET: t2_dir_thr_120
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler


KeyboardInterrupt: 

## **Interpretación detallada de la comparación GRU vs XGBoost**



1. Coincidencia entre modelos

Los dos modelos coinciden en aproximadamente el 85.1% de los casos. Esto significa que, en términos prácticos, ambos están leyendo una señal muy parecida la mayor parte del tiempo.

La consecuencia directa es que GRU y XGBoost no están aportando información completamente distinta. No parecen ser dos modelos complementarios en sentido fuerte, sino dos formas diferentes de aproximar casi la misma frontera de decisión.

2. Diferencia real entre modelos

Solo discrepan en el 14.9% de las observaciones, es decir, en 14,048 casos sobre 93,990.

Ese 14.9% es justamente la parte más interesante, porque ahí se ve si uno de los dos aporta valor adicional frente al otro.

En esos desacuerdos:

- GRU acierta en 28.8% de los casos
- XGBoost acierta en 36.7% de los casos
- ambos fallan en 34.4% de los casos

Esto muestra que, cuando los modelos piensan distinto, XGBoost resuelve mejor el conflicto. En otras palabras, en la zona difícil del problema, XGBoost toma mejores decisiones que GRU.

3. Qué está haciendo cada modelo en la práctica

La matriz de combinaciones muestra que ambos modelos predicen con mucha frecuencia la clase 0, es decir, la clase neutra:

- ambos predicen 0 al mismo tiempo en 50,865 casos
- eso representa más del 54% de toda la muestra

Además:

- ambos predicen -1 en 15,696 casos
- ambos predicen +1 en 13,381 casos

Esto confirma que ambos modelos están fuertemente alineados con la estructura del target T2: una gran parte del tiempo identifican neutralidad, y una menor proporción del tiempo identifican movimientos relevantes hacia arriba o hacia abajo.

4. Qué tan confiables son cuando coinciden

Cuando ambos coinciden en una clase, la precisión cambia bastante según la clase:

- acuerdo en -1: precisión ≈ 28.7%
- acuerdo en 0: precisión ≈ 71.6%
- acuerdo en +1: precisión ≈ 32.3%

Esto tiene una interpretación muy clara:

a. Cuando ambos dicen 0, suelen tener razón con bastante frecuencia.
Esto significa que los modelos son relativamente buenos detectando ausencia de movimiento fuerte.

b. Cuando ambos dicen -1 o +1, la precisión es bastante menor.
Esto implica que detectar movimientos significativos sigue siendo la parte difícil del problema.

En términos operativos, el modelo es mejor filtrando ruido que acertando direcciones extremas con alta pureza.

5. Implicancia para trading

Esto sugiere una interpretación muy importante:

El sistema, tal como está, no debe entenderse todavía como un generador de entradas directas de alta convicción en largo o corto. Más bien, funciona mejor como un filtro de contexto:

- identifica bastante bien cuándo no hay señal fuerte
- identifica de forma moderada cuándo podría haber una señal alcista o bajista
- pero todavía comete bastantes errores en las clases extremas

En otras palabras, el mayor valor actual del modelo parece estar en evitar operar condiciones neutras o ruidosas, más que en acertar con mucha precisión todos los movimientos fuertes.

6. Sobre el posible ensemble GRU + XGBoost

Dado que:

- la coincidencia entre ambos es muy alta
- XGBoost gana más veces cuando discrepan
- no aparece una complementariedad fuerte

la evidencia actual no sugiere que combinar ambos modelos vaya a producir una mejora importante de forma automática.

La lectura más razonable es:

- GRU no está agregando mucha señal nueva respecto a XGBoost
- XGBoost domina la comparación
- si hubiera que quedarse con un modelo base, hoy debería ser XGBoost

7. Conclusión práctica

La conclusión más sólida es la siguiente:

- ambos modelos capturan casi la misma señal
- XGBoost la captura mejor
- la principal fortaleza del sistema es detectar neutralidad
- la detección de señales alcistas y bajistas todavía es moderada, no contundente
- por lo tanto, el siguiente paso lógico no es combinar modelos, sino convertir XGBoost en reglas operativas más selectivas

8. Próximo paso más útil

Con estos resultados, lo más sólido sería analizar XGBoost por nivel de convicción, por ejemplo:

- probabilidad máxima de clase
- solo tomar casos donde predice +1 o -1 con alta confianza
- medir cómo cambia la precisión al filtrar por confianza

Ese análisis te diría si realmente puedes transformar esta señal en una regla de trading más limpia y menos ruidosa.